# AValidate · Audio: frozen BEATs vs the CNN baseline

BEATs is Microsoft's pretrained audio transformer. We keep it **frozen**, extract one 768-number embedding per second for every clip, and train only a small head on top, with the same loss, class weights, schedule, selection rule and metrics as `cnn_v4`. The only change is the encoder.

Two checkpoints, because AVE clips come from AudioSet:
- **iter3** (main result): self-supervised only; heard AudioSet audio but never its labels.
- **iter3_as2m** (comparison): pre-training also used AudioSet labels, so it may be optimistic on AVE.

**One-time:** download BEATs_iter3 from Microsoft's OneDrive link (see step 2) into `MyDrive/AValidate/pretrained/`. Needs a GPU runtime and the `GH_TOKEN` / `WANDB_API_KEY` secrets as before.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the runtime to GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
from google.colab import userdata

BRANCH = 'audio/beats'   # switch to 'main' once this branch is merged
REPO = '/content/Audio-Video-Confidence'
token = userdata.get('GH_TOKEN')
url = f'https://{token}@github.com/NUS-AIS-GCs/Audio-Video-Confidence.git'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO], check=True)
else:
    # fetch + hard reset (not pull): always matches GitHub exactly, even if branch history was rewritten
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
!git log --oneline -1

In [ ]:
!pip -q install -r requirements-audio.txt

In [ ]:
# Weights & Biases: reads WANDB_API_KEY from Colab Secrets (key icon). Skipped if the secret is missing.
WANDB_ENTITY = 'GC2-AValidate'   # team workspace on wandb.ai (shared with the team)
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.setdefault('WANDB_PROJECT', 'avalidate')
    WANDB = '--wandb'
    print('W&B on:', os.environ['WANDB_ENTITY'] + '/' + os.environ['WANDB_PROJECT'])
except Exception as e:
    WANDB = ''
    print('W&B off:', e)

## 1 · Data and audio
Same as the baseline notebook: AVE from your Drive, 16 kHz wavs on the local disk (about 9 minutes per fresh session).

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.data_ave

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.extract_audio

## 2 · BEATs checkpoints
**iter3** must be downloaded by hand once (OneDrive blocks scripts): open https://1drv.ms/u/s!AqeByhGUtINrgcpxJUNDxg4eU0r-vA?e=qezPJ5 , download, and upload the file to `MyDrive/AValidate/pretrained/` keeping a name that starts with `BEATs_iter3`. **iter3_as2m** downloads automatically (361 MB) and is checked against its sha256.

In [ ]:
import glob
print('pretrained folder:', sorted(os.path.basename(p) for p in glob.glob('/content/drive/MyDrive/AValidate/pretrained/*')))

## 3 · Extract per-second BEATs embeddings
Runs the frozen encoder once over every clip; cached on Drive (`cache/beats_iter3`, `cache/beats_iter3_as2m`), so later sessions skip it. A few minutes each on an L4.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.beats_extract --ckpt iter3

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.beats_extract --ckpt iter3_as2m

## 4 · Sanity check
The head should memorise 32 clips: **val sec-acc about 0.95 or higher** by the end (macro-F1 stays low here; ignore it).

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.train --features beats_iter3 --overfit 32 --epochs 40 --run-id overfit_beats

## 5 · Train the head: iter3 (main result)
Same settings as `cnn_v4`: 30 epochs, cosine learning rate, sqrt class weights, best epoch by validation per-second macro-F1.

In [ ]:
%cd /content/Audio-Video-Confidence
RUN_ID = 'beats_iter3_v1'
WANDB = globals().get('WANDB', '')
if not WANDB: print('W&B is OFF: run the W&B cell first if you want tracking')
!python -m audio.train --run-id {RUN_ID} --features beats_iter3 {WANDB}

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.evaluate --run-id {RUN_ID} {WANDB}
from IPython.display import Image, display
display(Image(f'/content/drive/MyDrive/AValidate/artifacts/audio_baseline/{RUN_ID}/confusion_test.png'))

## 6 · Comparison: iter3_as2m (saw AudioSet labels)

In [ ]:
%cd /content/Audio-Video-Confidence
RUN_ID_B = 'beats_as2m_v1'
!python -m audio.train --run-id {RUN_ID_B} --features beats_iter3_as2m {WANDB}

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.evaluate --run-id {RUN_ID_B} {WANDB}
display(Image(f'/content/drive/MyDrive/AValidate/artifacts/audio_baseline/{RUN_ID_B}/confusion_test.png'))

## 7 · Side by side

In [ ]:
import json, pandas as pd
rows = {}
for run in ['cnn_v4', 'beats_iter3_v1', 'beats_as2m_v1']:
    p = f'/content/drive/MyDrive/AValidate/artifacts/audio_baseline/{run}/results.json'
    if os.path.exists(p):
        rows[run] = json.load(open(p))['test']
pd.DataFrame(rows).T.drop(columns='n_clips').round(3)

## 8 · Export embeddings for the team
From the main result (iter3): `MyDrive/AValidate/embeddings/audio_beats_iter3_v1/{split}_emb.npy` (N, 10, 256) + `{split}_ids.csv`.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.export_embeddings --run-id beats_iter3_v1